# ESTA Style Analysis — Gemma 4 Server

Runs `gemma4:e4b` on Colab T4 GPU and exposes it via ngrok for ESTA-v2's style-analysis skill.

**This notebook is executed programmatically by Claude Code via colab-proxy-mcp.**
You do not need to run cells manually — the style-analysis skill does it for you.

If you need to run manually for debugging:
1. Runtime → Change runtime type → T4 GPU
2. Run cells 1 → 2 → 3 → 4 → 5 → 6 in order
3. Paste your ngrok token into Cell 2 (free at https://dashboard.ngrok.com)

In [ ]:
# Cell 1 — Check GPU
!nvidia-smi

Fri Apr 17 07:05:40 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   31C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
# Cell 2 — Install Ollama + Python packages
# NGROK_TOKEN injected by the style-analysis skill from config.yaml — do not hardcode here.
!apt-get install -qq zstd pciutils
!curl -fsSL https://ollama.com/install.sh | sh
!pip install -q pyngrok ollama

NGROK_TOKEN = ""  # injected at runtime by skill

from pyngrok import ngrok
ngrok.set_auth_token(NGROK_TOKEN)
print('Done')

In [ ]:
# Cell 3 — Start Ollama server
import subprocess, time, os

env = os.environ.copy()
env['OLLAMA_HOST'] = '0.0.0.0:11434'
env['CUDA_VISIBLE_DEVICES'] = '0'

subprocess.Popen(
    ['ollama', 'serve'],
    env=env,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)
time.sleep(5)
print('Ollama server started')

Ollama server started


In [ ]:
# Cell 4 — Pull model + create JSON alias + smoke test
import subprocess, ollama

BASE_MODEL = 'gemma4:e4b'

# Pull
print(f'Pulling {BASE_MODEL}...')
r = subprocess.run(['ollama', 'pull', BASE_MODEL], capture_output=True, text=True)
if r.returncode != 0:
    raise RuntimeError('Pull failed: ' + r.stderr[-300:])
print(f'{BASE_MODEL} ready')

# Create JSON-tuned alias (temperature=0 + system prompt baked in server-side)
MODEL = BASE_MODEL + '-json'
modelfile = f"""FROM {BASE_MODEL}
PARAMETER temperature 0
PARAMETER repeat_penalty 1.0
PARAMETER num_ctx 32768
SYSTEM "You are a structured-output assistant. Always respond with valid JSON exactly matching the schema provided. Do not include any explanation, preamble, or markdown. Output only the JSON object."
"""
with open('/tmp/Modelfile', 'w') as f:
    f.write(modelfile)

print(f'Creating {MODEL}...')
r = subprocess.run(
    ['ollama', 'create', MODEL, '-f', '/tmp/Modelfile'],
    capture_output=True, text=True
)
if r.returncode != 0:
    print('Create failed:', r.stderr[-200:])
    MODEL = BASE_MODEL
    print(f'Falling back to {MODEL}')
else:
    print(f'{MODEL} ready')

# Smoke test
print('\nTesting format enforcement...')
schema = {
    'type': 'object',
    'properties': {'status': {'type': 'string'}},
    'required': ['status']
}
resp = ollama.chat(
    model=MODEL,
    messages=[{'role': 'user', 'content': 'Reply with status ok'}],
    format=schema,
)
out = (resp.message.content or '').strip()
print(f'Output: {out!r}')
if out.startswith('{'):
    print('✅ Grammar enforcement working')
else:
    print('❌ Grammar enforcement NOT working')

print(f'\nModel: {MODEL}')

Pulling gemma4:e4b...
gemma4:e4b ready
Creating gemma4:e4b-json...
gemma4:e4b-json ready

Testing format enforcement...
Output: '{"status": "ok"}'
✅ Grammar enforcement working

Model: gemma4:e4b-json


In [ ]:
# Cell 5 — Expose via ngrok + emit URL for skill to parse
from pyngrok import ngrok
import urllib.request, json

tunnel = ngrok.connect(11434)
public_url = tunnel.public_url

# Skill parses this line to extract the URL and write it to config.yaml
print(f'OLLAMA_URL={public_url}')

try:
    r = urllib.request.urlopen('http://localhost:11434/api/tags', timeout=5)
    models = [m['name'] for m in json.loads(r.read()).get('models', [])]
    print(f'Models: {models}')
except Exception as e:
    print(f'Health check failed: {e}')

In [ ]:
# Test with the full AgentOutput schema via direct local call
import ollama, json

# Minimal AgentOutput-like schema with anyOf actions
schema = {
    "type": "object",
    "properties": {
        "current_state": {"type": "object", "properties": {"evaluation_previous_goal": {"type": "string"}, "memory":
{"type": "string"}, "next_goal": {"type": "string"}}, "required": ["evaluation_previous_goal", "memory",
"next_goal"]},
        "action": {
            "type": "array",
            "items": {
                "anyOf": [
                    {"type": "object", "properties": {"go_to_url": {"type": "object", "properties": {"url": {"type":
"string"}}, "required": ["url"]}}, "required": ["go_to_url"]},
                    {"type": "object", "properties": {"click_element_by_index": {"type": "object", "properties":
{"index": {"type": "integer"}}, "required": ["index"]}}, "required": ["click_element_by_index"]}
                ]
            }
        }
    },
    "required": ["current_state", "action"]
}

resp = ollama.chat(
    model='gemma4:e4b-json',
    messages=[{'role': 'user', 'content': 'Navigate to google.com'}],
    format=schema,
)
out = (resp.message.content or '').strip()
print(out)
print('✅ JSON' if out.startswith('{') else '❌ plain text')

{"current_state": {"evaluation_previous_goal": "Navigation to google.com", "memory": "The user requested navigation to google.com.", "next_goal": "Navigate to google.com (Action: Navigate)"}, "action": [{"go_to_url": {"url": "https://www.google.com"}}]}
✅ JSON


In [ ]:
import ollama

schema_with_refs = {
    "type": "object",
    "$defs": {
        "ActionItem": {
            "type": "object",
            "properties": {"url": {"type": "string"}},
            "required": ["url"]
        }
    },
    "properties": {
        "action": {
            "type": "array",
            "items": {"$ref": "#/$defs/ActionItem"}
        }
    },
    "required": ["action"]
}

resp = ollama.chat(
    model='gemma4:e4b-json',
    messages=[{'role': 'user', 'content': 'navigate to google.com'}],
    format=schema_with_refs,
)
out = (resp.message.content or '').strip()
print(out)
print('✅ JSON' if out.startswith('{') else '❌ plain text')

{"action": [] }
✅ JSON


In [ ]:
# Cell 6 — Keep-alive (leave this running)
# Colab disconnects after ~90 min idle. Re-run if you reconnect.
import time, urllib.request

print(f'Live at: {public_url}/v1')
print('Interrupt kernel to stop')

i = 0
while True:
    time.sleep(60)
    i += 1
    try:
        urllib.request.urlopen('http://localhost:11434/', timeout=3)
        print(f'  [{i}m] alive — {public_url}/v1')
    except Exception as e:
        print(f'  [{i}m] OLLAMA DOWN — restart Cell 3')

Live at: https://intromissive-lennon-processionally.ngrok-free.dev/v1
Interrupt kernel to stop
  [1m] alive — https://intromissive-lennon-processionally.ngrok-free.dev/v1


KeyboardInterrupt: 

In [ ]:
# Check model info via API
import urllib.request, json
r = urllib.request.urlopen('http://localhost:11434/api/show',
    data=json.dumps({'name': 'gemma4:e4b-json'}).encode())
info = json.loads(r.read())
print(json.dumps(info.get('model_info', {}), indent=2))
print('---')
# Also check what num_ctx is set to
details = info.get('details', {})
print('details:', details)

{
  "gemma4.attention.head_count": 8,
  "gemma4.attention.head_count_kv": 2,
  "gemma4.attention.key_length": 512,
  "gemma4.attention.key_length_swa": 256,
  "gemma4.attention.layer_norm_rms_epsilon": 1e-06,
  "gemma4.attention.shared_kv_layers": 18,
  "gemma4.attention.sliding_window": 512,
  "gemma4.attention.sliding_window_pattern": null,
  "gemma4.attention.value_length": 512,
  "gemma4.attention.value_length_swa": 256,
  "gemma4.audio.attention.head_count": 8,
  "gemma4.audio.attention.layer_norm_epsilon": 1e-06,
  "gemma4.audio.block_count": 12,
  "gemma4.audio.conv_kernel_size": 5,
  "gemma4.audio.embedding_length": 1024,
  "gemma4.audio.feed_forward_length": 4096,
  "gemma4.block_count": 42,
  "gemma4.context_length": 131072,
  "gemma4.embedding_length": 2560,
  "gemma4.embedding_length_per_layer_input": 256,
  "gemma4.feed_forward_length": 10240,
  "gemma4.final_logit_softcapping": 30,
  "gemma4.rope.dimension_count": 512,
  "gemma4.rope.dimension_count_swa": 256,
  "gemma4.r

In [ ]:
import urllib.request, json
r = urllib.request.urlopen('http://localhost:11434/api/show',
    data=json.dumps({'name': 'gemma4:e4b-json'}).encode())
info = json.loads(r.read())
params = info.get('parameters', '')
print('parameters:', params)
print('num_ctx in params:', 'num_ctx' in params)

parameters: repeat_penalty                 1
temperature                    0
top_k                          64
top_p                          0.95
num_ctx in params: False
